# Session 2 · Part 1 — Prepare paired graphs for DGAT

**Goal:** turn aligned, normalized RNA and protein measurements into the two graph inputs used during
training. A spatial transcriptomics prediction sample has RNA only; paired spatial CITE-seq training
samples provide both modalities. This notebook uses the same official Breast RNA/ADT pair as Session 1,
so the tensor dimensions and graph construction correspond to the real tutorial dataset.


In [ ]:
from pathlib import Path
import sys

current = Path.cwd().resolve()
for candidate in (current, *current.parents):
    if (candidate / "src" / "dgat_tutorial").is_dir():
        tutorial_root = candidate
        break
else:
    raise FileNotFoundError("Start Jupyter inside the hands-on_tutorial directory.")

sys.path.insert(0, str(tutorial_root / "src"))

from dgat_tutorial.checkpoints import tutorial_paths, write_checkpoint

paths = tutorial_paths(tutorial_root)
print(f"Tutorial root: {paths.root}")


## 1. Process paired modalities


In [ ]:
import numpy as np
import pandas as pd

from dgat_tutorial.data import find_dgat_h5ad_pair, load_tutorial_data
from dgat_tutorial.processing import knn_edge_index, process_modalities

pair = find_dgat_h5ad_pair(paths.raw_data)
dataset = load_tutorial_data(paths.raw_data)
processed = process_modalities(
    dataset.spots,
    dataset.transcripts.select_dtypes(include=[np.number]),
    dataset.proteins.select_dtypes(include=[np.number]),
)
source = f"RNA={pair[0]}, ADT={pair[1]}"


## 2. Construct the RNA and protein graph inputs


In [ ]:
# The official pipeline stores these as PyTorch Geometric HeteroData node/edge types.
# Here we expose the arrays first so dimensions and alignment are easy to inspect.
x_rna = processed.normalized_transcripts.to_numpy(dtype=np.float32)
x_protein = processed.normalized_proteins.to_numpy(dtype=np.float32)
rna_edge_index = knn_edge_index(processed.spots, n_neighbors=6)
protein_edge_index = knn_edge_index(processed.spots, n_neighbors=6)

graph_summary = pd.DataFrame([
    {"graph": "RNA", "nodes": len(x_rna), "node_features": x_rna.shape[1], "directed_edges": rna_edge_index.shape[1]},
    {"graph": "protein", "nodes": len(x_protein), "node_features": x_protein.shape[1], "directed_edges": protein_edge_index.shape[1]},
])
graph_summary


## 3. Understand the training and inference handoff

During training, paired RNA/protein graphs produce two latent representations. During inference, only
the RNA graph is encoded; its latent representation is sent through the trained protein decoder.

`RNA graph → RNA encoder → shared latent → protein decoder → predicted proteins`


In [ ]:
ids_path = paths.processed_data / "aligned_spot_ids.csv"
summary_path = paths.results / "session02_graph_input_summary.csv"
edge_path = paths.processed_data / "rna_spatial_edge_index.csv"
pd.DataFrame({"spot_id": processed.spots.index}).to_csv(ids_path, index=False)
graph_summary.assign(source=source).to_csv(summary_path, index=False)
pd.DataFrame(rna_edge_index.T, columns=["source_index", "target_index"]).to_csv(edge_path, index=False)
manifest = write_checkpoint(
    "2.1", [ids_path, summary_path, edge_path],
    summary={"source": source, "spots": len(processed.spots), "genes": x_rna.shape[1], "proteins": x_protein.shape[1]},
    start=paths.root,
)
print(f"Checkpoint written: {manifest}")


## Check

Both graphs must have the same node count and ordering in paired training data. Feature counts differ:
RNA nodes carry genes and protein nodes carry ADTs. Edge arrays use integer node positions and have
shape `2 × number_of_edges`.
